# ML-04 — Search Intelligence Data Contract

**Lane:** Refresh / Content Opportunity Scoring  
**Decision:** Which content pages should an editor review first?  
**Development window:** March 2026 features → April 2026 outcome  
**Sealed final month:** June 2026 is not used for label development.

This notebook keeps the contract simple: define the row, verify the warehouse slice with real queries, build only five safe features, then deliberately add one leaked feature to show why future information must be excluded.

In [2]:
%pip -q install duckdb huggingface_hub scikit-learn pandas numpy

import os
import getpass
import duckdb
import numpy as np
import pandas as pd

# Get the token without hard-coding it into this public notebook.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (input is hidden): ")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Keep the token out of the CREATE SECRET SQL text.
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])
con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN getvariable('hf_token'))"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"
APR = f"read_parquet('{FACT}/month=2026-04/*.parquet')"

print("Connected.")
print("Feature window: March 2026")
print("Outcome window: April 2026")
print("June 2026 remains sealed.")

Connected.
Feature window: March 2026
Outcome window: April 2026
June 2026 remains sealed.


In [3]:
from huggingface_hub import HfApi
from google.colab import userdata

token = userdata.get("HF_TOKEN")
api = HfApi(token=token)

me = api.whoami()
print("Logged in as:", me["name"])

files = api.list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=token
)

march_files = [f for f in files if "fact_content_daily_performance/month=2026-03" in f]
print("March files found:", len(march_files))
print(march_files[:5])

Logged in as: jaberahmad555
March files found: 1
['fact_content_daily_performance/month=2026-03/data_0.parquet']


## 1. Unit of analysis + time window

### Five plain-word contract answers

1. **What one row means:**  
   In the source table, one row is one **content item for one report date for one client**.  
   In my modeling frame, one row becomes **one content item aggregated over March 2026**.

2. **Which table I use:**  
   `fact_content_daily_performance`.

3. **Time window:**  
   March 2026 is the **feature month**. April 2026 is the **next-month outcome window**. June 2026 stays sealed and is not used to develop the label.

4. **What I predict/rank:**  
   I rank pages by the risk that their April impressions fall by at least **20%** compared with March. The proxy label is `is_declining_next_month`.

5. **What I deliberately exclude:**  
   Any **April outcome information** is excluded from the real feature set because it is not knowable at the March decision moment. GA4 measures are also left out of this first five-feature frame so the contract stays focused on search signals and does not silently depend on uneven analytics availability.

**Human output:** a ranked review queue that helps an editor decide which pages to inspect first.

## 2. Fields: feature / label / context / excluded

### Feature — safe at the decision moment
Exactly five features:

| Feature | Meaning | Available when? |
|---|---|---|
| `impressions_mar` | Total March GSC impressions | Knowable on April 1 because March is finished |
| `clicks_mar` | Total March GSC clicks | Knowable on April 1 because March is finished |
| `ctr_mar` | March clicks ÷ March impressions | Computed only from March values |
| `avg_position_mar` | Mean March GSC position when position > 0 | Computed only from March values |
| `active_days_mar` | Number of March dates with > 0 impressions | Computed only from March values |

### Label / proxy
`is_declining_next_month = 1` when April impressions are less than 80% of March impressions; otherwise `0`.

### Context — never model features
`client_hash_id`, `content_hash_id`.

### Excluded
- April impressions / any April-derived ratio → future outcome information.
- `future_ratio` → deliberately created only for the leakage demonstration, then deleted.
- GA4 measures → not used in this first feature frame because GA4 availability is uneven across the panel.

This is a **decision-support proxy**, not proof that refreshing a page will cause performance to improve.

## 3. Verify it with exactly three queries

The three checks below prove:  
1) grain, 2) slice size + date span, and 3) availability using `IS TRUE`.

In [4]:
# QUERY 1 — GRAIN CHECK
# If this returns zero rows, the claimed source grain holds for March.

grain_check = con.sql(f'''
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS rows_at_grain
    FROM {MAR}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 10
''').df()

print("QUERY 1 — duplicate rows at claimed grain:", len(grain_check))
display(grain_check)

QUERY 1 — duplicate rows at claimed grain: 0


,report_date,client_hash_id,content_hash_id,rows_at_grain


In [5]:
# QUERY 2 — ROW COUNT + DATE SPAN

slice_check = con.sql(f'''
    SELECT
        COUNT(*) AS march_rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {MAR}
''').df()

print("QUERY 2 — March slice size and date span")
display(slice_check)

QUERY 2 — March slice size and date span


,march_rows,clients,content_items,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


In [6]:
# QUERY 3 — AVAILABILITY CHECK
# Assignment requirement: filter with IS TRUE and show how many rows survive.

availability_check = con.sql(f'''
    SELECT
        COUNT(*) AS total_march_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_ga4_available,
        COUNT(DISTINCT content_hash_id)
            FILTER (WHERE ga4_data_available IS TRUE) AS content_items_ga4_available,
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*),
            2
        ) AS pct_rows_ga4_available
    FROM {MAR}
''').df()

print("QUERY 3 — GA4 availability (IS TRUE)")
display(availability_check)

QUERY 3 — GA4 availability (IS TRUE)


,total_march_rows,rows_ga4_available,content_items_ga4_available,pct_rows_ga4_available
0,9841378,413966,90489,4.21


### Build the five-feature frame

To avoid extremely unstable percentage changes, the modeling universe keeps pages with at least **100 March impressions**. March is the only feature window. April is used only to construct the next-month proxy label.

In [7]:
feature_frame = con.sql(f'''
    WITH mar_agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_mar,
            SUM(gsc_clicks) AS clicks_mar,
            CASE
                WHEN SUM(gsc_impressions) > 0
                THEN 1.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
                ELSE NULL
            END AS ctr_mar,
            AVG(
                CASE WHEN gsc_avg_position > 0
                     THEN gsc_avg_position
                END
            ) AS avg_position_mar,
            COUNT(DISTINCT CASE
                WHEN gsc_impressions > 0 THEN report_date
            END) AS active_days_mar
        FROM {MAR}
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100
    ),
    apr_agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_apr
        FROM {APR}
        GROUP BY 1, 2
    )
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.impressions_mar,
        m.clicks_mar,
        m.ctr_mar,
        m.avg_position_mar,
        m.active_days_mar,
        COALESCE(a.impressions_apr, 0) AS impressions_apr,
        CASE
            WHEN COALESCE(a.impressions_apr, 0) < 0.80 * m.impressions_mar
            THEN 1 ELSE 0
        END AS is_declining_next_month
    FROM mar_agg m
    LEFT JOIN apr_agg a
      USING (client_hash_id, content_hash_id)
''').df()

feature_cols = [
    "impressions_mar",
    "clicks_mar",
    "ctr_mar",
    "avg_position_mar",
    "active_days_mar",
]

print("Feature frame rows:", len(feature_frame))
print("Five safe features:", feature_cols)
print(
    "Declining-next-month base rate:",
    f"{feature_frame['is_declining_next_month'].mean():.2%}"
)
display(feature_frame.head(10))

Feature frame rows: 101441
Five safe features: ['impressions_mar', 'clicks_mar', 'ctr_mar', 'avg_position_mar', 'active_days_mar']
Declining-next-month base rate: 51.75%


,client_hash_id,content_hash_id,impressions_mar,clicks_mar,ctr_mar,avg_position_mar,active_days_mar,impressions_apr,is_declining_next_month
0,client_62f4a7e64f5e0096,content_76c1f31e2b38f054,747.0,1.0,0.001339,29.377302,31,249.0,1
1,client_62f4a7e64f5e0096,content_ffc5ab4b34aab1f8,501.0,2.0,0.003992,17.519486,31,281.0,1
2,client_62f4a7e64f5e0096,content_9739856fc83dc1ca,2893.0,1.0,0.000346,9.445005,31,722.0,1
3,client_62f4a7e64f5e0096,content_3d1dc691a3502105,6955.0,9.0,0.001294,4.365083,31,3068.0,1
4,client_62f4a7e64f5e0096,content_9d28af4f99c5e67b,6789.0,3.0,0.000442,7.602191,31,3896.0,1
5,client_62f4a7e64f5e0096,content_612f75f9482a1955,6412.0,7.0,0.001092,9.260566,31,8300.0,0
6,client_62f4a7e64f5e0096,content_49dcc118cfb555d0,822.0,1.0,0.001217,17.730958,31,2922.0,0
7,client_62f4a7e64f5e0096,content_5cd4254950e4a883,21906.0,24.0,0.001096,4.897889,31,34438.0,0
8,client_62f4a7e64f5e0096,content_29437b06bda4410e,1526.0,1.0,0.000655,25.904136,31,439.0,1
9,client_62f4a7e64f5e0096,content_c00e33641178f421,19818.0,19.0,0.000959,4.997863,31,15819.0,1


### The trap: deliberately leak the future, then remove it

`future_ratio = April impressions / March impressions` directly contains the same future information used to define the label. It is **not available at the March decision moment**.

I add it once on purpose to show how an unrealistically strong score can appear. Then I delete it and keep only the honest five-feature result.

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

model_df = feature_frame.dropna(
    subset=feature_cols + ["is_declining_next_month"]
).copy()

X_honest = model_df[feature_cols]
y = model_df["is_declining_next_month"].astype(int)
groups = model_df["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.25, random_state=42
)
train_idx, test_idx = next(splitter.split(X_honest, y, groups=groups))

def precision_at_k(y_true, scores, k=50):
    y_arr = np.asarray(y_true)
    s_arr = np.asarray(scores)
    k = min(k, len(y_arr))
    top = np.argsort(s_arr)[::-1][:k]
    return float(y_arr[top].mean())

def fit_and_score(X):
    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    )
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    probs = model.predict_proba(X.iloc[test_idx])[:, 1]
    auc = roc_auc_score(y.iloc[test_idx], probs)
    p50 = precision_at_k(y.iloc[test_idx], probs, k=50)
    return auc, p50

honest_auc, honest_p50 = fit_and_score(X_honest)

# DELIBERATE LEAK: this column uses April outcome information.
model_df["future_ratio"] = (
    model_df["impressions_apr"] / model_df["impressions_mar"]
)

X_leaked = model_df[feature_cols + ["future_ratio"]]
leaked_auc, leaked_p50 = fit_and_score(X_leaked)

print(f"HONEST model — ROC-AUC: {honest_auc:.3f} | Precision@50: {honest_p50:.3f}")
print(f"LEAKED model — ROC-AUC: {leaked_auc:.3f} | Precision@50: {leaked_p50:.3f}")
print("The leaked score should jump sharply because future_ratio contains the answer.")

# Remove the trap and keep the honest feature set.
model_df = model_df.drop(columns=["future_ratio"])
final_feature_cols = feature_cols.copy()

assert "future_ratio" not in model_df.columns
assert len(final_feature_cols) == 5

print("\nLeak removed.")
print("Final safe feature set:", final_feature_cols)
print(f"Final honest Precision@50 kept: {honest_p50:.3f}")

HONEST model — ROC-AUC: 0.620 | Precision@50: 0.580
LEAKED model — ROC-AUC: 1.000 | Precision@50: 1.000
The leaked score should jump sharply because future_ratio contains the answer.

Leak removed.
Final safe feature set: ['impressions_mar', 'clicks_mar', 'ctr_mar', 'avg_position_mar', 'active_days_mar']
Final honest Precision@50 kept: 0.580


## 4. Data limits

**Named limitation:** this label measures an **observed next-month drop in search impressions**. It does **not** prove that a content refresh would cause performance to recover.

Other caution: client histories and GA4 availability are uneven across the panel, so one March→April experiment should not be treated as universal evidence. The final June month remains sealed for later validation.

In [9]:
# Small supporting limitation check: client coverage inside the final modeling frame.
# This is not one of the three contract-verification queries above; it only describes
# the already-built small feature frame and does not scan the warehouse again.

coverage_summary = pd.DataFrame({
    "model_rows": [len(model_df)],
    "clients_in_model_frame": [model_df["client_hash_id"].nunique()],
    "positive_label_rate": [model_df["is_declining_next_month"].mean()],
})

display(coverage_summary)

,model_rows,clients_in_model_frame,positive_label_rate
0,101441,44,0.517463


## 5. Self-check

After **Runtime → Run all** succeeds, confirm:

- [x] Every section above is filled — markdown thinking **and** code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] Exactly three verification queries have visible outputs
- [x] Availability is checked using `ga4_data_available IS TRUE`
- [x] The final feature frame has exactly five safe features
- [x] The deliberate leakage experiment is shown, then the leaked column is removed
- [x] No client names, URLs, private queries, or Hugging Face token are exposed
- [x] Claims use careful words: observed, proxy, directional, decision-support
- [x] Committed as `work/notebooks/w03_data_contract.ipynb`
- [x] Submitted the main public repo URL on the ML-04 card
